# 이광수 문체 StyleModel — EXAONE-3.5-2.4B QLoRA

**목표:** 평이한 현대 한국어 입력 → 1930~40년대 이광수(춘원) 문체로 재서술하는 LoRA 어댑터 학습.

**실행 환경:** Colab 무료 T4(16GB)에서 동작하도록 4비트(QLoRA) 구성.

**준비물:** 로컬에서 만든 `data/pairs/pairs.jsonl` (현재 **4,620쌍** 준비됨).

**순서:** 런타임 → 런타임 유형 변경 → T4 GPU 선택 후, 셀을 위에서부터 차례로 실행.

In [ ]:
# 1) 의존성 설치
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "bitsandbytes>=0.43" accelerate datasets
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2) pairs.jsonl 업로드
아래 셀 실행 후, 로컬 `new-version-multiagent/data/pairs/pairs.jsonl` 을 선택해 업로드.

In [ ]:
from google.colab import files
up = files.upload()  # pairs.jsonl 선택
PAIRS = list(up.keys())[0]
print('업로드:', PAIRS)

In [ ]:
# 3) 데이터 로드 + 채팅 포맷 구성
from datasets import load_dataset
from transformers import AutoTokenizer

# 네이티브 지원 모델(커스텀 코드 불필요) — 가장 안정적.
BASE = 'Qwen/Qwen2.5-3B-Instruct'
# 더 작고 빠른 것: 'Qwen/Qwen2.5-1.5B-Instruct'
# EXAONE는 최신 transformers와 remote code 충돌(get_input_embeddings)로 비권장.

SYSTEM = ('너는 1930~40년대 소설가 이광수(춘원)다. 입력으로 주어진 평이한 현대 한국어 문장을, '
          '이광수 특유의 근대 국어 문체로 다시 써라. 한자어·격식체 어미·예스러운 어휘를 살리고 '
          '뜻은 그대로 보존하라. 변환한 문장만 출력하라.')

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(ex):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': ex['neutral']},
        {'role': 'assistant', 'content': ex['lgs']},
    ]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

ds = load_dataset('json', data_files=PAIRS, split='train')
ds = ds.map(to_text, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=0.03, seed=42)
print(ds)
print('--- 예시 ---')
print(ds['train'][0]['text'][:600])

In [ ]:
# 4) 4비트 모델 로드 (QLoRA)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # bf16: 학습 안정(스케일러 불필요)
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto',
    trust_remote_code=True, torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print('loaded', model.config.model_type)

In [ ]:
# 5) LoRA + SFTTrainer 설정 (T4 최적화)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir='lgs_style_lora',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,                # bf16: 손실 스케일러 안 써서 dtype 에러 없음 (안정)
    max_length=256,
    dataset_text_field='text',
    packing=False,            # T4는 FlashAttention 미지원 → packing 끄기
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=cfg, peft_config=lora,
    train_dataset=ds['train'], eval_dataset=ds['test'],
)
trainer.train()

In [ ]:
# 6) 말투만 평가 — 친일과 무관한 일상 문장으로 '순수 문체 변환'만 확인
#    (내용/논증은 나중에 KnowledgeAgent 담당. 이 모델은 '근대 말투' 입히기만 본다)
import torch
tests = [
    # --- 완전 일상 문장: 말투 일반화 테스트 (친일과 전혀 무관) ---
    '나는 아침에 일찍 일어나 밥을 먹고 천천히 길을 걸었다.',
    '그 사람은 약속을 잘 지키고 늘 성실하게 일한다.',
    '비가 와서 우산을 들고 시장에 갔는데 사람이 무척 많았다.',
    '친구와 오랜만에 만나 차를 마시며 옛이야기를 나누었다.',
    '봄이 되니 마당의 나무에 새 잎이 돋고 꽃이 피었다.',
    # --- 이광수 원문 파생(참고 비교용) ---
    '한 시대를 지배하던 생각이 저물고 새로운 생각이 그 자리를 차지하려 할 때, 사람들은 늘 개혁을 떠올린다.',
]
model.config.use_cache = True
model.eval()
for t in tests:
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': t}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=True,
                             temperature=0.8, top_p=0.9)
    gen = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    print('[입력]', t)
    print('[근대체]', gen.strip())
    print('-' * 60)

In [ ]:
# 7) 어댑터 저장 + 다운로드 (로컬 추론용)
trainer.model.save_pretrained('lgs_style_lora_final')
tok.save_pretrained('lgs_style_lora_final')
!zip -r lgs_style_lora_final.zip lgs_style_lora_final >/dev/null
from google.colab import files
files.download('lgs_style_lora_final.zip')
print('완료: lgs_style_lora_final.zip 다운로드. 50~200MB 어댑터를 로컬로 가져가 4비트 추론에 사용.')